# Training regressors with MultiTrain

This walkthrough uses the housing dataset included with the repository. We will predict `price` and keep the measurements from every built-in regressor so the complete result can be inspected.

## Import MultiTrain and load the dataset

As in the classifier example, the installed package version is printed and saved with the notebook output.

In [1]:
from pathlib import Path

import pandas as pd

import MultiTrain
from MultiTrain import MultiRegressor

print(f"MultiTrain version: {MultiTrain.__version__}")

dataset_path = Path("examples/datasets/Housing.csv")
df = pd.read_csv(dataset_path)
df.head()

MultiTrain version: 1.2.0


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## Inspect the columns

The dataset mixes numerical housing measurements with several categorical yes/no columns and a furnishing-status column. MultiTrain will encode those categorical values after creating the holdout set.

In [2]:
print(f"Dataset shape: {df.shape}")
print(f"Missing values: {int(df.isna().sum().sum())}")
print()
print("Column types:")
print(df.dtypes)

Dataset shape: (545, 13)
Missing values: 0

Column types:
price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object


## Configure MultiTrain and split the data

`custom_models` remains `None`, so all built-in regressors are included. Two model workers keep the example reasonably quick without letting each estimator consume every CPU core.

In [3]:
train = MultiRegressor(
    n_jobs=1,
    model_workers=2,
    random_state=42,
    max_iter=300,
)

regression_split = train.split(
    data=df,
    target="price",
    test_size=0.3,
    random_state=42,
    auto_cat_encode=True,
)

X_train, X_test, y_train, y_test = regression_split
print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
X_train.head()

Training features: (381, 12)
Test features: (164, 12)


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
126,7160,3,1,1,0,0,0,0,0,2,0,0
363,3584,2,1,1,0,0,1,1,0,0,1,1
370,4280,2,1,1,0,0,1,0,1,2,1,1
31,7000,3,1,4,0,0,1,0,1,2,1,1
113,9620,3,1,1,0,0,0,0,0,2,0,2


## Train and measure every regressor

The results are ordered by mean absolute error, where lower values appear first. Training measurements are included beside the test measurements so you can inspect the differences yourself.

In [4]:
pd.set_option("display.max_rows", None)

regression_results = train.fit(
    datasplits=regression_split,
    show_train_score=True,
    sort="mean_absolute_error",
)

regression_results

Training Models:   0%|          | 0/37 [00:00<?, ?it/s]

,mean_absolute_error,mean_squared_error_train,mean_squared_error,r2_score_train,r2_score,mean_absolute_error_train,median_absolute_error_train,median_absolute_error,mean_squared_log_error_train,mean_squared_log_error,explained_variance_score_train,explained_variance_score,root_mean_squared_error_train,root_mean_squared_error,Time
MLPRegressor,876251.326198,754999509743.487549,1413037435150.642578,0.759674,0.671874,634792.278582,469052.668394,667719.338094,0.031664,0.057591,0.759684,0.673701,868907.077738,1188712.511565,73.67ms
SVR,877281.937887,632235215557.777588,1510640426345.020996,0.798752,0.64921,486529.358151,231565.874003,626385.643957,0.022356,0.063052,0.80213,0.649588,795132.200051,1229081.130904,32.82ms
NuSVR,886844.111578,630194877429.215576,1520534028849.717529,0.799401,0.646912,525646.420695,363338.889229,628660.815145,0.022689,0.063658,0.802788,0.647309,793848.145069,1233099.358872,27.44ms
CatBoostRegressor,892997.534574,144431019402.807129,1454391943108.743408,0.954026,0.662271,296490.638638,231560.453064,671271.61371,0.008724,0.060854,0.954026,0.66251,380040.812812,1205981.734152,400.63ms
PoissonRegressor,915697.374583,912796193175.467163,1508204048619.161865,0.709445,0.649775,692263.460796,487643.100847,673430.569312,0.036804,0.062898,0.709445,0.651615,955403.680742,1228089.593075,3.38ms
SGDRegressor,923652.625726,966901927017.691284,1524000696712.029541,0.692223,0.646107,725271.088589,537222.309044,688879.231707,0.039272,0.061866,0.692227,0.648019,983311.714065,1234504.231144,5.98ms
LinearSVR,925447.90677,965154449409.735352,1534973715494.053955,0.692779,0.643559,719359.570628,531650.547931,674650.286744,0.038642,0.06234,0.692779,0.645314,982422.744754,1238940.561728,4.25ms
LinearRegression,925543.548316,965153171508.67334,1535047758428.049805,0.69278,0.643542,719440.739875,532853.656148,675166.140564,0.038649,0.062342,0.69278,0.645302,982422.094371,1238970.442919,1.39ms
Lars,925543.548316,965153171508.673218,1535047758428.050537,0.69278,0.643542,719440.739875,532853.656148,675166.140564,0.038649,0.062342,0.69278,0.645302,982422.094371,1238970.442919,2.04ms
Lasso,925543.985092,965153171557.930786,1535050120826.395752,0.69278,0.643541,719440.001196,532850.437103,675156.126129,0.038649,0.062342,0.69278,0.645302,982422.094396,1238971.396291,1.58ms


## Confirm the run

This check reports how many regressors returned results and whether any model was unable to produce all of the standard test measurements.

In [5]:
regression_metrics = [
    "mean_squared_error",
    "r2_score",
    "mean_absolute_error",
    "median_absolute_error",
    "mean_squared_log_error",
    "explained_variance_score",
    "root_mean_squared_error",
]
failed_regressors = regression_results[regression_metrics].isna().all(axis=1)

print(f"Models returned: {len(regression_results)}")
print(f"Models with every test metric missing: {failed_regressors.sum()}")
if failed_regressors.any():
    print(regression_results.index[failed_regressors].tolist())

Models returned: 37
Models with every test metric missing: 0
